# Manual Loss Calculation for Any Model Type
How to manually calculate and track losses for sklearn, Keras, and any other ML models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras.optimizers import Adam
from keras.regularizers import L2
from keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

## Section 1: Load and Prepare Data

In [ ]:
df = pd.read_csv('../data/lab09/data.csv')

# Prepare features and target
X = df[['R_q','F_q','M_q','RFM_score']].values
y = df['price'].values

# Scale target for better loss values
ss_X = StandardScaler()
ss_y = StandardScaler()

X_scaled = ss_X.fit_transform(X)
y_scaled = ss_y.fit_transform(y.reshape(-1, 1)).flatten()

# Split into train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_scaled, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## Section 2: sklearn Models with Manual Loss Tracking

The key difference: **sklearn models don't track losses automatically. You calculate them yourself after training.**

In [ ]:
# Train sklearn models
models_sklearn = {
  'LinearRegression': LinearRegression(),
  'Ridge (L2=0.01)': Ridge(alpha=0.01),
  'SVR': SVR(kernel='rbf', C=100)
}

# Dictionary to store losses
sklearn_losses = {}

for name, model in models_sklearn.items():
    # Train
    model.fit(X_train, y_train)
    
    # Get predictions on train and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Calculate losses manually
    train_mse = mean_squared_error(y_train, y_train_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    
    # Store results
    sklearn_losses[name] = {
        'train_mse': train_mse,
        'val_mse': val_mse,
        'train_mae': train_mae,
        'val_mae': val_mae
    }
    
    print(f"\n{name}")
    print(f"  Train MSE: {train_mse:.4f} | Val MSE: {val_mse:.4f}")
    print(f"  Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")
    print(f"  Gap (MSE): {val_mse - train_mse:.4f}")

## Section 3: Manual Epoch-wise Loss Tracking for sklearn

For iterative sklearn models (e.g., SGDRegressor), you can manually track losses across iterations:

In [ ]:
from sklearn.linear_model import SGDRegressor

# Manual tracking for iterative model
losses_history = {'epoch': [], 'train_mse': [], 'val_mse': [], 'train_mae': [], 'val_mae': []}

sgd = SGDRegressor(max_iter=1, warm_start=True, random_state=42)

for epoch in range(20):
    # Partial fit one epoch at a time
    sgd.fit(X_train, y_train)
    
    # Calculate losses
    train_mse = mean_squared_error(y_train, sgd.predict(X_train))
    val_mse = mean_squared_error(y_val, sgd.predict(X_val))
    train_mae = mean_absolute_error(y_train, sgd.predict(X_train))
    val_mae = mean_absolute_error(y_val, sgd.predict(X_val))
    
    # Store
    losses_history['epoch'].append(epoch + 1)
    losses_history['train_mse'].append(train_mse)
    losses_history['val_mse'].append(val_mse)
    losses_history['train_mae'].append(train_mae)
    losses_history['val_mae'].append(val_mae)

# Convert to DataFrame for easy viewing
history_df = pd.DataFrame(losses_history)
print("SGDRegressor training history:")
print(history_df.head(10))

## Section 4: Keras Models with Automatic History (for comparison)

In [ ]:
keras_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, 'relu', kernel_regularizer=L2(0.01)),
    Dropout(0.3),
    Dense(32, 'relu', kernel_regularizer=L2(0.01)),
    Dropout(0.3),
    Dense(1)
])

keras_model.compile(optimizer=Adam(), loss='mse', metrics=['mae'])

# Keras automatically tracks losses in history object
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
keras_history = keras_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print("Keras history keys:", keras_history.history.keys())
print(f"Number of epochs trained: {len(keras_history.history['loss'])}")

## Section 5: Side-by-Side Comparison: sklearn vs Keras

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: sklearn models - MSE comparison
ax = axes[0, 0]
for name, losses in sklearn_losses.items():
    ax.scatter([1, 2], [losses['train_mse'], losses['val_mse']], s=100, label=name, alpha=0.7)
    ax.plot([1, 2], [losses['train_mse'], losses['val_mse']], '--', alpha=0.5)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Train', 'Validation'])
ax.set_ylabel('MSE')
ax.set_title('sklearn Models: MSE Comparison')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: sklearn models - MAE comparison
ax = axes[0, 1]
for name, losses in sklearn_losses.items():
    ax.scatter([1, 2], [losses['train_mae'], losses['val_mae']], s=100, label=name, alpha=0.7)
    ax.plot([1, 2], [losses['train_mae'], losses['val_mae']], '--', alpha=0.5)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Train', 'Validation'])
ax.set_ylabel('MAE')
ax.set_title('sklearn Models: MAE Comparison')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: SGD Epoch-wise MSE
ax = axes[1, 0]
ax.plot(history_df['epoch'], history_df['train_mse'], 'o-', label='Train MSE', linewidth=2)
ax.plot(history_df['epoch'], history_df['val_mse'], 's-', label='Val MSE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('SGDRegressor: Epoch-wise Loss Tracking')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Keras MSE
ax = axes[1, 1]
ax.plot(keras_history.history['loss'], 'o-', label='Train Loss', linewidth=2)
ax.plot(keras_history.history['val_loss'], 's-', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Keras Sequential: Loss Tracking')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY DIFFERENCES:")
print("="*60)
print("sklearn: Manual loss calculation after model.fit()")
print("Keras: Automatic loss tracking in history object")
print("="*60)

## Section 6: Universal Loss Tracking Framework

Here's a reusable utility class that works with any model:

In [ ]:
class LossTracker:
    """Universal loss tracker for any model type"""
    
    def __init__(self, model_name):
        self.model_name = model_name
        self.history = {
            'epoch': [], 'train_mse': [], 'val_mse': [],
            'train_mae': [], 'val_mae': []
        }
    
    def track(self, y_train, y_train_pred, y_val, y_val_pred, epoch=None):
        """Record losses for this epoch/iteration"""
        train_mse = mean_squared_error(y_train, y_train_pred)
        val_mse = mean_squared_error(y_val, y_val_pred)
        train_mae = mean_absolute_error(y_train, y_train_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        
        self.history['epoch'].append(epoch or len(self.history['epoch']) + 1)
        self.history['train_mse'].append(train_mse)
        self.history['val_mse'].append(val_mse)
        self.history['train_mae'].append(train_mae)
        self.history['val_mae'].append(val_mae)
    
    def plot(self, ax=None):
        """Plot the tracked losses"""
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 5))
        
        ax.plot(self.history['epoch'], self.history['train_mse'], 'o-', label='Train MSE', linewidth=2)
        ax.plot(self.history['epoch'], self.history['val_mse'], 's-', label='Val MSE', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss')
        ax.set_title(f'{self.model_name}: Loss Tracking')
        ax.legend()
        ax.grid(alpha=0.3)
        return ax
    
    def get_gap(self):
        """Return bias-variance gap"""
        final_train = self.history['train_mse'][-1]
        final_val = self.history['val_mse'][-1]
        return final_val - final_train

# Example: Use with any model
tracker = LossTracker('Random Forest')

# Simulate an iterative training process
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=42)

# For non-iterative models, just track once after training
rf.fit(X_train, y_train)
y_train_pred = rf.predict(X_train)
y_val_pred = rf.predict(X_val)
tracker.track(y_train, y_train_pred, y_val, y_val_pred, epoch=1)

# Compare with test set
y_test_pred = rf.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
print(f"\nRandom Forest Results:")
print(f"Train MSE: {tracker.history['train_mse'][0]:.4f}")
print(f"Val MSE: {tracker.history['val_mse'][0]:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Bias-Variance Gap: {tracker.get_gap():.4f}")

## Section 7: Quick Reference - How to Calculate Losses for Any Model

```
# For ANY model (sklearn, XGBoost, LightGBM, etc.):

1. Train: model.fit(X_train, y_train)

2. Predict:
   y_train_pred = model.predict(X_train)
   y_val_pred = model.predict(X_val)

3. Calculate losses:
   train_mse = mean_squared_error(y_train, y_train_pred)
   val_mse = mean_squared_error(y_val, y_val_pred)
   train_mae = mean_absolute_error(y_train, y_train_pred)
   val_mae = mean_absolute_error(y_val, y_val_pred)

4. Analyze bias-variance gap:
   gap = val_mse - train_mse
   # gap > 0.2 → overfitting
   # gap ≈ 0.05 → good generalization
```

**For iterative models (SGDRegressor, GradientBoosting, etc.):**

Loop through iterations, recalculating predictions and losses at each step to track convergence.